In [1]:
import cv2
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, f1_score
import joblib
import torch
import torch.nn as nn
from torchvision.transforms import Resize, ToTensor, Compose, Normalize, transforms

from tqdm import tqdm
from torchvision import transforms
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from os import listdir
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from keras.layers import Reshape
from keras.layers import Input, Dense, concatenate
from keras.models import Model
from collections import defaultdict
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from torchvision.models import mobilenet_v3_large
from sklearn.metrics import precision_score, recall_score
import seaborn as sns
import tensorflow as tf

2026-03-03 16:37:44.475272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772555864.744424      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772555864.820246      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772555865.492460      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772555865.492514      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772555865.492517      17 computation_placer.cc:177] computation placer alr

In [2]:
data = pd.read_csv("/kaggle/input/datasets/sealeopard/brain-tumor-csv/Brain Tumor.csv")
data.head()

,Image,Class,Mean,Variance,Standard Deviation,Entropy,Skewness,Kurtosis,Contrast,Energy,ASM,Homogeneity,Dissimilarity,Correlation,Coarseness
0,Image1,0,6.535339,619.587845,24.891522,0.109059,4.276477,18.900575,98.613971,0.293314,0.086033,0.530941,4.473346,0.981939,7.458341e-155
1,Image2,0,8.749969,805.957634,28.389393,0.266538,3.718116,14.464618,63.858816,0.475051,0.225674,0.651352,3.220072,0.988834,7.458341e-155
2,Image3,1,7.341095,1143.808219,33.820234,0.001467,5.061750,26.479563,81.867206,0.031917,0.001019,0.268275,5.981800,0.978014,7.458341e-155
3,Image4,1,5.958145,959.711985,30.979219,0.001477,5.677977,33.428845,151.229741,0.032024,0.001026,0.243851,7.700919,0.964189,7.458341e-155
4,Image5,0,7.315231,729.540579,27.010009,0.146761,4.283221,19.079108,174.988756,0.343849,0.118232,0.501140,6.834689,0.972789,7.458341e-155


In [3]:
images = data.Image
classes = data.Class
image_label_dict = data.set_index('Image')['Class'].to_dict()

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
def preprocess_image(image_path):
  image = Image.open(image_path).convert('RGB')
  transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      #transforms.Normalize(mean=[0.5], std=[0.5])
  ])
  image = transform(image)
  image = image.unsqueeze(0)
  return image.to(device)
def extract_features(image, model):
  with torch.no_grad():
    feature = model(image).squeeze().cpu().numpy()
  return feature
def extract_features_from_image(image_path, model):
  image = preprocess_image(image_path)
  features = extract_features(image, model)
  return features
def preprocess_image_pca(image_path):
    image = Image.open(image_path).convert('RGB') 
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    image = transform(image)
    image = image.numpy().flatten()
    return image

In [6]:
import torchvision.models as models

mobilenet_v3_large_model = models.mobilenet_v3_large(pretrained=True) 
mobilenet_v3_large_model = nn.Sequential(*list(mobilenet_v3_large_model.children())[:-1])  
mobilenet_v3_large_model = mobilenet_v3_large_model.to(device)
mobilenet_v3_large_model.eval()

dir_list = '/kaggle/input/datasets/sealeopard/reloaded-crop/Crop/'
X_mobilenet_v3_large = []
X_pca_orig = []
Y = []
filename_list = []

for filename in tqdm(listdir(dir_list)):
    if filename.endswith('.jpg'):
        filename_list.append(filename)
        path = dir_list + filename
        label = image_label_dict.get(filename.split('.')[0], None)
        
        if label is not None:
            features = extract_features_from_image(path, mobilenet_v3_large_model)
            features = features.flatten()  
            X_mobilenet_v3_large.append(features)
            feature_pca = preprocess_image_pca(path)
            X_pca_orig.append(feature_pca)
            Y.append(label)

X_resnet = np.array(X_mobilenet_v3_large)
X_pca_orig = np.array(X_pca_orig)
Y = np.array(Y)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 21.1M/21.1M [00:00<00:00, 127MB/s] 
100%|██████████| 3762/3762 [02:08<00:00, 29.31it/s]


In [7]:
print(X_resnet[0].shape)
print(X_pca_orig[0].shape)

(960,)
(150528,)


In [8]:
n_components = 0.95
pca_model = PCA(n_components=n_components, random_state=42)
X_pca = pca_model.fit_transform(X_pca_orig)
print(X_pca[0].shape)
X_pca = np.hstack((X_resnet, X_pca))
print(X_pca[0].shape)

(378,)
(1338,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, Y, test_size=0.5, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
print(X_train.shape)

(1598, 1338)


In [10]:
def create_cnn_model(input_shape, num_filters):
  model = Sequential()
  model = Sequential()
  model.add(Reshape((input_shape[1], 1), input_shape=input_shape[1:]))
  model.add(Conv1D(filters=num_filters, kernel_size=3, activation='relu', input_shape=(500, 1)))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Conv1D(filters=num_filters * 2, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Flatten())
  model.add(Dense(64, activation='relu'))
  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

model1 = create_cnn_model(X_train.shape, 32)
model2 = create_cnn_model(X_train.shape, 64)
model3 = create_cnn_model(X_train.shape, 48)

#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_val = scaler.transform(X_val)
#_test = scaler.transform(X_test)

X_train = np.array(X_train)
X_val = np.array(X_val)
X_test = np.array(X_test)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

X_train = X_train.astype('float32')
X_val = X_val.astype('float32')
X_test = X_test.astype('float32')

y_train = y_train.astype('float32')
y_val = y_val.astype('float32')
y_test = y_test.astype('float32')

model1.fit(X_train, y_train, epochs=40, batch_size=32, validation_data=(X_val, y_val))
model2.fit(X_train, y_train, epochs=40, batch_size=32, validation_data=(X_val, y_val))
model3.fit(X_train, y_train, epochs=40, batch_size=32, validation_data=(X_val, y_val))

loss, accuracy = model1.evaluate(X_test, y_test)
print(f'Accuracy1: {accuracy*100:.2f}%')

loss, accuracy = model2.evaluate(X_test, y_test)
print(f'Accuracy2: {accuracy*100:.2f}%')

loss, accuracy = model3.evaluate(X_test, y_test)
print(f'Accuracy3: {accuracy*100:.2f}%')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-03 16:44:34.479464: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.6604 - loss: 0.8065 - val_accuracy: 0.9081 - val_loss: 0.2406
Epoch 2/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.8935 - loss: 0.2428 - val_accuracy: 0.9435 - val_loss: 0.1684
Epoch 3/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9592 - loss: 0.1337 - val_accuracy: 0.9470 - val_loss: 0.1465
Epoch 4/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9578 - loss: 0.1142 - val_accuracy: 0.9470 - val_loss: 0.1700
Epoch 5/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.9761 - loss: 0.0701 - val_accuracy: 0.9470 - val_loss: 0.1581
Epoch 6/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9921 - loss: 0.0311 - val_accuracy: 0.9470 - val_loss: 0.1616
Epoch 7/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9907 - loss: 0.0367 - val_accuracy: 0.9435 - val_loss: 0.2186
Epoch 8/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9898 - loss: 0.0297 - val_accuracy: 0.9505 - v

In [11]:
y_pred1 = model1.predict(X_test)
y_pred2 = model2.predict(X_test)
y_pred3 = model3.predict(X_test)

thresholds = np.arange(0, 1.01, 0.01) 
best_threshold_1 = 0.0
best_threshold_2 = 0.0
best_threshold_3 = 0.0

best_f1 = 0.0

for threshold in thresholds:
  y_pred_classes = (y_pred1 >= threshold).astype(int)
  f1 = f1_score(y_test, y_pred_classes)
  if f1 > best_f1:
    best_f1 = f1
    best_threshold_1 = threshold
print(f'F1-score1: {best_f1:.4f}')
print('best threshold: ', best_threshold_1)
y_pred1 = (y_pred1 > best_threshold_1).astype(int)

best_threshold = 0.0
best_f1 = 0.0

for threshold in thresholds:
  y_pred_classes = (y_pred2 >= threshold).astype(int)
  f1 = f1_score(y_test, y_pred_classes)
  if f1 > best_f1:
    best_f1 = f1
    best_threshold_2 = threshold
print(f'F1-score2: {best_f1:.4f}')
print('best threshold: ', best_threshold_2)
y_pred2 = (y_pred2 > best_threshold_2).astype(int)

best_threshold = 0.0
best_f1 = 0.0

for threshold in thresholds:
  y_pred_classes = (y_pred3 >= threshold).astype(int)
  f1 = f1_score(y_test, y_pred_classes)
  if f1 > best_f1:
    best_f1 = f1
    best_threshold_3 = threshold
print(f'F1-score3: {best_f1:.4f}')
print('best threshold: ', best_threshold_3)
y_pred3 = (y_pred3 > best_threshold_3).astype(int)

59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
F1-score1: 0.9482
best threshold:  0.31
F1-score2: 0.9537
best threshold:  0.48
F1-score3: 0.9476
best threshold:  0.54


In [12]:
def print_metrics(y_test, y_pred, model_number):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Model {model_number} - Metrics:")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [13]:
import math
def fuzzy_rank(CF, top):
    R_L = np.zeros(CF.shape)
    for i in range(CF.shape[0]):
        for j in range(CF.shape[1]):
            for k in range(CF.shape[2]):
                R_L[i][j][k] = (2 - 1 * 2 ** CF[i][j][k])

    K_L = 1 * np.ones(shape=R_L.shape)
    for i in range(R_L.shape[0]):
        for sample in range(R_L.shape[1]):
            for k in range(top - 1):
                a = R_L[i][sample]
                idx = np.where(a == np.partition(a, k)[k])
                K_L[i][sample][idx] = R_L[i][sample][idx]

    return K_L

def CFS_func(CF, K_L):
    H = CF.shape[0]
    for f in range(CF.shape[0]):
        for i in range(CF.shape[1]):
            idx = np.where(K_L[f][i] == 1)
            CF[f][i][idx] = 0
    CFS = 1 - np.sum(CF, axis=0) / H
    return CFS

def Mitcherlich(top=2, *argv):
    L = len(argv)
    num_classes = argv[0].shape[1]
    CF = np.zeros(shape=(L, argv[0].shape[0], num_classes))

    for i, arg in enumerate(argv):
        CF[i] = arg

    R_L = fuzzy_rank(CF, top)
    RS = np.sum(R_L, axis=0)
    CFS = CFS_func(CF, R_L)
    FS = RS * CFS

    predictions = np.argmin(FS, axis=1) 
    return predictions
    
y_pred1 = model1.predict(X_test)
y_pred2 = model2.predict(X_test)
y_pred3 = model3.predict(X_test)

y_pred1 = np.hstack((1 - y_pred1, y_pred1))  
y_pred2 = np.hstack((1 - y_pred2, y_pred2))
y_pred3 = np.hstack((1 - y_pred3, y_pred3))

print(y_pred1.shape) 
print(y_pred2.shape) 
print(y_pred3.shape)  
predictions = Mitcherlich(2, y_pred1, y_pred2, y_pred3)
print_metrics(y_test, predictions, 'Ensemble Mitscherlich')

59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
(1881, 2)
(1881, 2)
(1881, 2)
Model Ensemble Mitscherlich - Metrics:
Accuracy: 95.69%
Precision: 0.9622
Recall: 0.9404
F1 Score: 0.9512

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.97      0.96      1042
         1.0       0.96      0.94      0.95       839

    accuracy                           0.96      1881
   macro avg       0.96      0.96      0.96      1881
weighted avg       0.96      0.96      0.96      1881



In [14]:
import json
import joblib
from tensorflow.keras.models import save_model   # hoặc model.save()

# ────────────────────────────────────────────────
# 1. Tạo thư mục lưu
# ────────────────────────────────────────────────
SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(pca_model,  os.path.join(SAVE_DIR, "pca_model.joblib"))

# ────────────────────────────────────────────────
# 3. Lưu 3 mô hình CNN (dạng Keras native format - khuyến nghị)
# ────────────────────────────────────────────────
model1.save(os.path.join(SAVE_DIR, "cnn_model_32filters.keras"))
model2.save(os.path.join(SAVE_DIR, "cnn_model_64filters.keras"))
model3.save(os.path.join(SAVE_DIR, "cnn_model_48filters.keras"))

print("Đã lưu 3 mô hình CNN (.keras format)")

# model1.save(os.path.join(SAVE_DIR, "cnn_model_32filters.h5"))
# model2.save(os.path.join(SAVE_DIR, "cnn_model_64filters.h5"))
# model3.save(os.path.join(SAVE_DIR, "cnn_model_48filters.h5"))

# ────────────────────────────────────────────────
# 4. Lưu metadata quan trọng (dùng để tái tạo inference chính xác)
# ────────────────────────────────────────────────
metadata = {
    "n_components": n_components,
    "image_size": (224, 224),
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225],
    "models": {
        "model1": {"filters_first": 32, "file": "cnn_model_32filters.keras"},
        "model2": {"filters_first": 64, "file": "cnn_model_64filters.keras"},
        "model3": {"filters_first": 48, "file": "cnn_model_48filters.keras"}
    },
    "best_thresholds": {
        # Bạn cần lưu lại best_threshold của từng mô hình sau khi tìm ở phần grid search
        "model1": best_threshold_1,   # thay bằng giá trị thật bạn tìm được
        "model2": best_threshold_2,
        "model3": best_threshold_3
    },
    "ensemble": {
        "method": "Gompertz",
        "top": 2
    },
    "classes": sorted(set(data['Class'])),  # hoặc list các class bạn có
    "date_trained": "2025-03"
}

with open(os.path.join(SAVE_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Đã lưu metadata.json")
print("Tất cả mô hình đã được lưu vào:", SAVE_DIR)

Đã lưu 3 mô hình CNN (.keras format)
Đã lưu metadata.json
Tất cả mô hình đã được lưu vào: /kaggle/working/saved_models
